# 10 — vLLM: Qasper Benchmark

This notebook evaluates KV cache compression on the
[Qasper](https://huggingface.co/datasets/tau/scrolls) (SCROLLS) benchmark
using [vLLM](https://github.com/vllm-project/vllm) with Qwen3-8B.

We test two compression strategies:
- **full_replacement** — full KV cache replacement at target ratio
- **filtering** — filtering-based compression at target ratio

Qasper contains questions over full NLP research papers (~3K–8K tokens),
testing document comprehension with both short and free-form answers.

Scoring uses the HuggingFace `evaluate` library (SQuAD F1), matching the
standard lm-evaluation-harness methodology for SCROLLS/Qasper.

Results are saved to `results/vllm_qasper/` for comparison in later notebooks.

## Configuration

In [1]:
MODEL_NAME = "Qwen/Qwen3-8B"

COMPRESSION_RATIOS = [0.01, 0.25, 0.50, 0.75]

FRACTION = 0.01

MAX_NEW_TOKENS = 64

PRESS_CONFIGS = {
    "full_replacement": lambda cr: {
        "model": MODEL_NAME,
        "dtype": "auto",
        "gpu_memory_utilization": 0.90,
        "trust_remote_code": True,
        "attention_config": {"backend": "FLASH_ATTN"},
        "kv_compression_algorithm": "full_replacement",
        "kv_compression_ratio": cr,
        "enable_prefix_caching": False,
    },
    "filtering": lambda cr: {
        "model": MODEL_NAME,
        "dtype": "auto",
        "gpu_memory_utilization": 0.90,
        "trust_remote_code": True,
        "attention_config": {"backend": "FLASH_ATTN"},
        "kv_compression_algorithm": "filtering",
        "kv_compression_ratio": cr,
    },
}

In [2]:
import sys
import builtins

_original_print = builtins.print

def print(*args, **kwargs):
    _original_print(*args, **kwargs)
    if sys.stdout is not sys.__stdout__:
        kwargs['file'] = sys.__stdout__
        kwargs['flush'] = True
        _original_print(*args, **kwargs)

In [3]:
import sys
import os

FORK_DIR = "/opt/app-root/src/vllm-fork"

if os.path.isdir(FORK_DIR) and os.listdir(FORK_DIR):
    sys.path.insert(0, FORK_DIR)
    import vllm
    print(f"Using FORK vLLM (version: {vllm.__version__})")
else:
    import vllm
    print(f"Using SYSTEM vLLM (version: {vllm.__version__})")

/opt/app-root/src/vllm-fork/vllm/__init__.py:7: RuntimeWarning: Failed to read commit hash:
No module named 'vllm._version'
  from .version import __version__, __version_tuple__  # isort:skip


Using FORK vLLM (version: dev)
Using FORK vLLM (version: dev)


In [4]:
import gc
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected.")

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
allocated_gb = torch.cuda.memory_allocated() / 1e9
reserved_gb = torch.cuda.memory_reserved() / 1e9

print(f"GPU:        {torch.cuda.get_device_name(0)}")
print(f"VRAM:       {vram_gb:.1f} GB total")
print(f"Allocated:  {allocated_gb:.2f} GB")
print(f"Reserved:   {reserved_gb:.2f} GB")
print(f"Free:       {vram_gb - reserved_gb:.1f} GB (approx)")

if allocated_gb > 1.0:
    print(
        "\n⚠  GPU memory is not free — a model from another notebook may still be loaded.\n"
        "   Restart this kernel before proceeding."
    )

def cleanup_vllm(llm):
    llm.llm_engine.engine_core.shutdown()
    del llm
    gc.collect()
    torch.cuda.empty_cache()

GPU:        NVIDIA A100-SXM4-40GB
VRAM:       42.4 GB total
Allocated:  0.00 GB
Reserved:   0.00 GB
Free:       42.4 GB (approx)
GPU:        NVIDIA A100-SXM4-40GB
VRAM:       42.4 GB total
Allocated:  0.00 GB
Reserved:   0.00 GB
Free:       42.4 GB (approx)


## 1. Load Qasper Dataset

In [5]:
from datasets import load_dataset

qasper_ds = load_dataset("tau/scrolls", "qasper", split="validation")
if FRACTION < 1.0:
    n = max(1, int(len(qasper_ds) * FRACTION))
    qasper_ds = qasper_ds.select(range(n))

print(f"Qasper dataset loaded: {len(qasper_ds)} examples")
print(f"Columns: {qasper_ds.column_names}")
print(f"Sample input (first 200 chars): {qasper_ds[0]['input'][:200]}...")
print(f"Sample output: {qasper_ds[0]['output']}")

/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Qasper dataset loaded: 17 examples
Columns: ['id', 'pid', 'input', 'output']
Sample input (first 200 chars): which multilingual approaches do they compare with?

Introduction
Although Neural Machine Translation (NMT) has dominated recent research on translation tasks BIBREF0, BIBREF1, BIBREF2, NMT heavily re...
Qasper dataset loaded: 17 examples
Columns: ['id', 'pid', 'input', 'output']
Sample output: BIBREF19, BIBREF20
Sample input (first 200 chars): which multilingual approaches do they compare with?

Introduction
Although Neural Machine Translation (NMT) has dominated recent research on translation tasks BIBREF0, BIBREF1, BIBREF2, NMT heavily re...
Sample output: BIBREF19, BIBREF20


## 2. Load Scoring Metric

In [6]:
import evaluate

squad_metric = evaluate.load("squad")
print("Loaded SQuAD metric (token-level F1)")

Loaded SQuAD metric (token-level F1)
Loaded SQuAD metric (token-level F1)


## 3. Prepare Prompts

Apply the model's chat template to each Qasper example.
The SCROLLS `input` field contains `question \n\n context`.

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

prompt_configs = []
max_input_tokens = 0

for row in qasper_ds:
    messages = [{"role": "user", "content": row["input"]}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )

    input_len = len(tokenizer.encode(prompt, add_special_tokens=False))
    max_input_tokens = max(max_input_tokens, input_len)

    references = row["output"].split(" | ") if " | " in row["output"] else [row["output"]]

    prompt_configs.append({
        "prompt": prompt,
        "reference_answers": references,
        "id": row["id"],
    })

print(f"Prepared {len(prompt_configs)} prompts")
print(f"Max input tokens: {max_input_tokens}")

Prepared 17 prompts
Max input tokens: 5748
Prepared 17 prompts
Max input tokens: 5748


## 4. Run Batch Inference

For each (algorithm, compression_ratio) combination, create a vLLM engine,
run all prompts in batch, and collect predictions.

In [8]:
import time
from vllm import LLM, SamplingParams

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_NEW_TOKENS,
)

def get_gpu_memory_used_gb() -> float:
    free, total = torch.cuda.mem_get_info()
    return (total - free) / 1e9

prompts = [pc["prompt"] for pc in prompt_configs]

all_results = []

configs = [("no_press", 0.0, {"model": MODEL_NAME, "dtype": "auto",
    "gpu_memory_utilization": 0.90,
    "trust_remote_code": True,
    "attention_config": {"backend": "FLASH_ATTN"}})]
for press_name, press_factory in PRESS_CONFIGS.items():
    for ratio in COMPRESSION_RATIOS:
        engine_args = press_factory(ratio)
        configs.append((press_name, ratio, engine_args))

for press_name, ratio, engine_args in configs:
    label = f"{press_name} | ratio={ratio}"
    print(f"\n{'='*60}")
    print(f"Running: {label} ({len(prompts)} prompts)")
    print(f"{'='*60}")

    llm = None
    try:
        llm = LLM(**engine_args)

        mem_before = get_gpu_memory_used_gb()
        start = time.perf_counter()

        outputs = llm.generate(prompts, sampling_params)

        batch_elapsed = time.perf_counter() - start
        mem_after = get_gpu_memory_used_gb()
        peak_mem = max(mem_before, mem_after)

        for i, output in enumerate(outputs):
            predicted_answer = output.outputs[0].text.strip()
            pc = prompt_configs[i]

            all_results.append({
                "framework": "vllm",
                "press": press_name,
                "compression_ratio": ratio,
                "predicted_answer": predicted_answer,
                "reference_answers": pc["reference_answers"],
                "id": pc["id"],
                "elapsed_sec": round(batch_elapsed / len(prompts), 3),
                "peak_gpu_mem_gb": round(peak_mem, 3),
            })

        total_gen_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
        throughput = total_gen_tokens / batch_elapsed if batch_elapsed > 0 else 0

        print(f"  Done: {batch_elapsed:.1f}s — {throughput:.1f} tok/s — peak mem={peak_mem:.2f} GB")
    finally:
        if llm is not None:
            cleanup_vllm(llm)

print(f"\nTotal results: {len(all_results)}")


Running: no_press | ratio=0.0 (17 prompts)

Running: no_press | ratio=0.0 (17 prompts)
INFO 08-31 14:50:26 [utils.py:233] non-default args: {'trust_remote_code': True, 'disable_log_stats': True, 'attention_config': AttentionConfig(backend=<AttentionBackendEnum.FLASH_ATTN: 'vllm.v1.attention.backends.flash_attn.FlashAttentionBackend'>, flash_attn_version=None, use_prefill_decode_attention=False, flash_attn_max_num_splits_for_cuda_graph=32, use_cudnn_prefill=False, use_trtllm_ragged_deepseek_prefill=False, use_trtllm_attention=None, disable_flashinfer_prefill=True, disable_flashinfer_q_quantization=False, use_prefill_query_quantization=False), 'model': 'Qwen/Qwen3-8B'}
INFO 08-31 14:50:26 [model.py:533] Resolved architecture: Qwen3ForCausalLM
INFO 08-31 14:50:26 [model.py:1582] Using max model len 40960
INFO 08-31 14:50:26 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 08-31 14:50:26 [vllm.py:795] Asynchronous scheduling is enabled.
WARNING 08-31 14

/opt/app-root/src/vllm-fork/vllm/__init__.py:7: RuntimeWarning: Failed to read commit hash:
No module named 'vllm._version'
  from .version import __version__, __version_tuple__  # isort:skip


(EngineCore pid=110069) INFO 08-31 14:50:37 [core.py:103] Initializing a V1 LLM engine (vdev) with config: model='Qwen/Qwen3-8B', speculative_config=None, tokenizer='Qwen/Qwen3-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=40960, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_c

(EngineCore pid=110069) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=110069) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  20% Completed | 1/5 [00:01<00:04,  1.02s/it]
Loading safetensors checkpoint shards:  40% Completed | 2/5 [00:02<00:03,  1.06s/it]
Loading safetensors checkpoint shards:  60% Completed | 3/5 [00:03<00:02,  1.08s/it]
Loading safetensors checkpoint shards:  80% Completed | 4/5 [00:04<00:01,  1.01s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:04<00:00,  1.28it/s]
Loading safetensors checkpoint shards: 100% Complet

(EngineCore pid=110069) INFO 08-31 14:50:45 [default_loader.py:384] Loading weights took 4.51 seconds
(EngineCore pid=110069) INFO 08-31 14:50:45 [gpu_model_runner.py:4627] Model loading took 15.27 GiB memory and 5.507707 seconds
(EngineCore pid=110069) INFO 08-31 14:50:50 [backends.py:988] Using cache directory: /opt/app-root/src/.cache/vllm/torch_compile_cache/f1781885f0/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=110069) INFO 08-31 14:50:50 [backends.py:1048] Dynamo bytecode transform time: 4.21 s
(EngineCore pid=110069) INFO 08-31 14:50:51 [backends.py:284] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.310 s
(EngineCore pid=110069) INFO 08-31 14:50:51 [monitor.py:48] torch.compile took 5.86 s in total
(EngineCore pid=110069) INFO 08-31 14:50:51 [decorators.py:296] Directly load AOT compilation from path /opt/app-root/src/.cache/vllm/torch_compile_cache/torch_aot_compile/b5b87fb51ca98babdcdb46b974078c98a405315eff9cadba9be7225f8

...

(EngineCore pid=112453) INFO 08-31 14:57:26 [default_loader.py:384] Loading weights took 4.47 seconds
(EngineCore pid=112453) INFO 08-31 14:57:27 [gpu_model_runner.py:4627] Model loading took 15.27 GiB memory and 5.463620 seconds
(EngineCore pid=112453) INFO 08-31 14:57:31 [backends.py:988] Using cache directory: /opt/app-root/src/.cache/vllm/torch_compile_cache/f1781885f0/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=112453) INFO 08-31 14:57:31 [backends.py:1048] Dynamo bytecode transform time: 4.18 s
(EngineCore pid=112453) INFO 08-31 14:57:33 [backends.py:284] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.301 s
(EngineCore pid=112453) INFO 08-31 14:57:33 [monitor.py:48] torch.compile took 5.83 s in total
(EngineCore pid=112453) INFO 08-31 14:57:33 [decorators.py:296] Directly load AOT compilation from path /opt/app-root/src/.cache/vllm/torch_compile_cache/torch_aot_compile/b5b87fb51ca98babdcdb46b974078c98a405315eff9cadba9be7225f8

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 22.32it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 27.06it/s]


(EngineCore pid=112453) INFO 08-31 14:57:42 [gpu_model_runner.py:5807] Graph capturing finished in 4 secs, took 0.52 GiB
(EngineCore pid=112453) INFO 08-31 14:57:42 [gpu_worker.py:617] CUDA graph pool memory: 0.52 GiB (actual), 0.52 GiB (estimated), difference: 0.0 GiB (0.8%).
(EngineCore pid=112453) INFO 08-31 14:57:42 [core.py:281] init engine (profile, create kv cache, warmup model) took 14.81 seconds
(EngineCore pid=112453) INFO 08-31 14:57:42 [vllm.py:795] Asynchronous scheduling is enabled.
INFO 08-31 14:57:42 [llm.py:391] Supported tasks: ['generate']


Processed prompts: 100%|██████████| 17/17 [00:31<00:00,  1.87s/it, est. speed input: 2682.98 toks/s, output: 34.19 toks/s]
[rank0]:[W831 14:58:15.631583751 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


  Done: 32.0s — 34.0 tok/s — peak mem=40.26 GB
  Done: 32.0s — 34.0 tok/s — peak mem=40.26 GB
(EngineCore pid=112453) INFO 08-31 14:58:14 [core.py:1201] Shutdown initiated (timeout=0)
(EngineCore pid=112453) INFO 08-31 14:58:14 [core.py:1224] Shutdown complete

Total results: 153

Total results: 153


## 5. Score & Results

Score predictions using the HuggingFace `evaluate` SQuAD metric (token-level F1).
For multi-reference answers (pipe-delimited in SCROLLS), we take the max F1
across references — standard SCROLLS evaluation methodology.

In [9]:
import pandas as pd

df = pd.DataFrame(all_results)

def compute_squad_f1(group):
    predictions = []
    references = []
    for _, row in group.iterrows():
        predictions.append({"id": str(row["id"]), "prediction_text": row["predicted_answer"]})
        references.append({"id": str(row["id"]), "answers": {
            "text": row["reference_answers"],
            "answer_start": [0] * len(row["reference_answers"]),
        }})
    result = squad_metric.compute(predictions=predictions, references=references)
    return result

all_metrics = {}
rows = []
for (press, ratio), group in df.groupby(["press", "compression_ratio"]):
    metrics = compute_squad_f1(group)
    key = f"{press}__{ratio}"
    all_metrics[key] = metrics
    rows.append({
        "press": press,
        "compression_ratio": ratio,
        "f1": round(metrics["f1"], 2),
        "exact_match": round(metrics["exact_match"], 2),
        "mean_time": round(group["elapsed_sec"].mean(), 3),
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

           press  compression_ratio   f1  exact_match  mean_time
       filtering               0.01 4.45          0.0      2.227
       filtering               0.25 6.60          0.0      1.998
       filtering               0.50 3.76          0.0      1.945
       filtering               0.75 2.91          0.0      1.881
full_replacement               0.01 4.45          0.0      0.489
full_replacement               0.25 4.91          0.0      0.467
full_replacement               0.50 3.50          0.0      0.452
full_replacement               0.75 2.10          0.0      0.437
        no_press               0.00 5.12          0.0      0.280
           press  compression_ratio   f1  exact_match  mean_time
       filtering               0.01 4.45          0.0      2.227
       filtering               0.25 6.60          0.0      1.998
       filtering               0.50 3.76          0.0      1.945
       filtering               0.75 2.91          0.0      1.881
full_replacement         

## 6. Save Results

In [10]:
import json
import os

os.makedirs("results/vllm_qasper", exist_ok=True)

predictions_path = "results/vllm_qasper/predictions.csv"
df.to_csv(predictions_path, index=False)
print(f"Saved predictions to {predictions_path}")

metrics_path = "results/vllm_qasper/metrics.json"
with open(metrics_path, "w") as f:
    json.dump(all_metrics, f, indent=2)
print(f"Saved metrics to {metrics_path}")

Saved predictions to results/vllm_qasper/predictions.csv
Saved predictions to results/vllm_qasper/predictions.csv
Saved metrics to results/vllm_qasper/metrics.json
Saved metrics to results/vllm_qasper/metrics.json
